<a href="https://colab.research.google.com/github/Sunil-143-as/Practical-1/blob/main/PRACTICAL_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# ============================================================
# 1. IMPORT REQUIRED MODULE
# ============================================================
from datetime import datetime
# ============================================================
# 2. FIREWALL CONFIGURATION
# ============================================================
firewall_enabled = True
# Default security policy
# If no rule matches, traffic will be BLOCKED.
default_policy = "BLOCK"
# ============================================================
# 3. FIREWALL RULE DATABASE
# ============================================================
firewall_rules = [
 # --------------------------------------------------------
 # INBOUND RULES
 # --------------------------------------------------------
 {
 "id": 1,
 "name": "Allow HTTP",
 "direction": "INBOUND",
 "source": "ANY",
 "destination": "LOCAL",
 "protocol": "TCP",
 "port": 80,
 "action": "ALLOW"
 },
 {
 "id": 2,
 "name": "Allow HTTPS",
 "direction": "INBOUND",
 "source": "ANY",
 "destination": "LOCAL",
 "protocol": "TCP",
 "port": 443,
 "action": "ALLOW"
 },
 {
 "id": 3,
 "name": "Allow SSH from Admin",
 "direction": "INBOUND",
 "source": "192.168.1.100",
 "destination": "LOCAL",
 "protocol": "TCP",
 "port": 22,
 "action": "ALLOW"
 },
 {
 "id": 4,
 "name": "Block Telnet",
 "direction": "INBOUND",
 "source": "ANY",
 "destination": "LOCAL",
 "protocol": "TCP",
 "port": 23,
 "action": "BLOCK"
 },
 {
 "id": 5,
 "name": "Block FTP",
 "direction": "INBOUND",
 "source": "ANY",
 "destination": "LOCAL",
 "protocol": "TCP",
 "port": 21,
 "action": "BLOCK"
 },
 # --------------------------------------------------------
 # OUTBOUND RULES
 # --------------------------------------------------------
 {
 "id": 6,
 "name": "Allow DNS",
 "direction": "OUTBOUND",
 "source": "LOCAL",
 "destination": "ANY",
 "protocol": "UDP",
 "port": 53,
 "action": "ALLOW"
 },
 {
 "id": 7,
 "name": "Allow HTTPS Outbound",
 "direction": "OUTBOUND",
 "source": "LOCAL",
 "destination": "ANY",
 "protocol": "TCP",
 "port": 443,
 "action": "ALLOW"
 },
 {
 "id": 8,
 "name": "Block Telnet Outbound",
 "direction": "OUTBOUND",
 "source": "LOCAL",
 "destination": "ANY",
 "protocol": "TCP",
 "port": 23,
 "action": "BLOCK"
 },
 {
 "id": 9,
 "name": "Allow DNS TCP",
 "direction": "OUTBOUND",
 "source": "LOCAL",
 "destination": "ANY",
 "protocol": "TCP",
 "port": 53,
 "action": "ALLOW"
 }
]
# ============================================================
# 4. FIREWALL LOG DATABASE
# ============================================================
firewall_logs = []
# ============================================================
# 5. DISPLAY FIREWALL STATUS
# ============================================================
def display_firewall_status():
    print("\n")
    print("=" * 80)
    print("FIREWALL STATUS")
    print("=" * 80)
    if firewall_enabled:
        print("Firewall Status : ENABLED")
    else:
        print("Firewall Status : DISABLED")
    print("Default Policy :", default_policy)
    print("=" * 80)
# ============================================================
# 6. DISPLAY FIREWALL RULES
# ============================================================
def display_firewall_rules():
    print("\n")
    print("=" * 120)
    print("CONFIGURED FIREWALL RULES")
    print("=" * 120)
    print(
        f"{'ID':<5}"
        f"{'RULE NAME':<27}"
        f"{'DIRECTION':<12}"
        f"{'SOURCE':<18}"
        f"{'PROTOCOL':<10}"
        f"{'PORT':<8}"
        f"{'ACTION':<10}"
    )
    print("-" * 120)
    for rule in firewall_rules:
        print(
            f"{rule['id']:<5}"
            f"{rule['name']:<27}"
            f"{rule['direction']:<12}"
            f"{rule['source']:<18}"
            f"{rule['protocol']:<10}"
            f"{rule['port']:<8}"
            f"{rule['action']:<10}"
        )
    print("=" * 120)
# ============================================================
# 7. IP ADDRESS MATCHING FUNCTION
# ============================================================
def ip_matches(rule_address, packet_address):
    # ANY means every address is accepted
    if rule_address.upper() == "ANY":
        return True
    # LOCAL represents the local computer
    if rule_address.upper() == "LOCAL":
        return packet_address.upper() == "LOCAL"
    # Exact IP comparison
    return rule_address == packet_address
# ============================================================
# 8. CHECK FIREWALL RULE
# ============================================================
def check_firewall(
    direction,
    source,
    destination,
    protocol,
    port
):
    direction = direction.upper()
    protocol = protocol.upper()
    # --------------------------------------------------------
    # Firewall disabled
    # --------------------------------------------------------
    if not firewall_enabled:
        decision = "ALLOW"
        rule_name = "FIREWALL DISABLED"
        create_log(
            source,
            destination,
            protocol,
            port,
            direction,
            decision,
            rule_name
        )
        return decision, rule_name
    # --------------------------------------------------------
    # Check every firewall rule
    # --------------------------------------------------------
    for rule in firewall_rules:
        # Check direction
        if rule["direction"] != direction:
            continue
        # Check protocol
        if rule["protocol"] != protocol:
            continue
        # Check port
        if rule["port"] != port:
            continue
        # Check source IP
        if not ip_matches(rule["source"], source):
            continue
        # Check destination IP
        if not ip_matches(rule["destination"], destination):
            continue
        # Matching rule found
        decision = rule["action"]
        rule_name = rule["name"]
        create_log(
            source,
            destination,
            protocol,
            port,
            direction,
            decision,
            rule_name
        )
        return decision, rule_name
    # --------------------------------------------------------
    # No rule found
    # Apply default policy
    # --------------------------------------------------------
    decision = default_policy
    rule_name = "DEFAULT DENY"
    create_log(
        source,
        destination,
        protocol,
        port,
        direction,
        decision,
        rule_name
    )
    return decision, rule_name
# ============================================================
# 9. CREATE FIREWALL LOG
# ============================================================
def create_log(
    source,
    destination,
    protocol,
    port,
    direction,
    decision,
    rule_name
):
    log_entry = {
        "time": datetime.now().strftime("%H:%M:%S"),
        "source": source,
        "destination": destination,
        "protocol": protocol,
        "port": port,
        "direction": direction,
        "decision": decision,
        "rule": rule_name
    }
    firewall_logs.append(log_entry)
# ============================================================
# 10. TEST NETWORK TRAFFIC
# ============================================================
def test_traffic(
    direction,
    source,
    destination,
    protocol,
    port
):
    print("\n")
    print("=" * 85)
    print("NETWORK TRAFFIC ANALYSIS")
    print("=" * 85)
    print("Direction :", direction)
    print("Source :", source)
    print("Destination :", destination)
    print("Protocol :", protocol)
    print("Port :", port)
    print("-" * 85)
    decision, rule_name = check_firewall(
        direction,
        source,
        destination,
        protocol,
        port
    )
    print("Matched Rule:", rule_name)
    print("Decision :", decision)
    if decision == "ALLOW":
        print("RESULT : TRAFFIC ALLOWED")
    else:
        print("RESULT : TRAFFIC BLOCKED")
    print("=" * 85)
# ============================================================
# 11. ADD FIREWALL RULE
# ============================================================
def add_firewall_rule(
    name,
    direction,
    source,
    destination,
    protocol,
    port,
    action
):
    new_id = 1
    if len(firewall_rules) > 0:
        new_id = max(
            rule["id"]
            for rule in firewall_rules
        ) + 1
    new_rule = {
        "id": new_id,
        "name": name,
        "direction": direction.upper(),
        "source": source,
        "destination": destination,
        "protocol": protocol.upper(),
        "port": port,
        "action": action.upper()
    }
    firewall_rules.append(new_rule)
    print("\nFirewall rule added successfully.")
    print("Rule ID :", new_rule["id"])
    print("Rule Name :", new_rule["name"])
    print("Direction :", new_rule["direction"])
    print("Protocol :", new_rule["protocol"])
    print("Port :", new_rule["port"])
    print("Action :", new_rule["action"])
# ============================================================
# 12. DELETE FIREWALL RULE
# ============================================================
def delete_firewall_rule(rule_id):
    for rule in firewall_rules:
        if rule["id"] == rule_id:
            firewall_rules.remove(rule)
            print("\nFirewall rule deleted:", rule["name"])
            return
    print("\nRule ID not found.")
# ============================================================
# 13. ENABLE FIREWALL
# ============================================================
def enable_firewall():
    global firewall_enabled
    firewall_enabled = True
    print("\nFirewall has been ENABLED.")
# ============================================================
# 14. DISABLE FIREWALL
# ============================================================
def disable_firewall():
    global firewall_enabled
    firewall_enabled = False
    print("\nFirewall has been DISABLED.")
# ============================================================
# 15. DISPLAY FIREWALL LOGS
# ============================================================
def display_logs():
    print("\n")
    print("=" * 125)
    print("FIREWALL ACTIVITY LOG")
    print("=" * 125)
    if len(firewall_logs) == 0:
        print("No traffic has been processed.")
        return
    print(
        f"{'TIME':<10}"
        f"{'SOURCE':<18}"
        f"{'DESTINATION':<18}"
        f"{'PROTOCOL':<10}"
        f"{'PORT':<8}"
        f"{'DIRECTION':<12}"
        f"{'DECISION':<10}"
        f"{'RULE'}"
    )
    print("-" * 125)
    for log in firewall_logs:
        print(
            f"{log['time']:<10}"
            f"{log['source']:<18}"
            f"{log['destination']:<18}"
            f"{log['protocol']:<10}"
            f"{log['port']:<8}"
            f"{log['direction']:<12}"
            f"{log['decision']:<10}"
            f"{log['rule']}"
        )
    print("=" * 125)
# ============================================================
# 16. FIREWALL STATISTICS
# ============================================================
def display_statistics():
    total = len(firewall_logs)
    allowed = 0
    blocked = 0
    for log in firewall_logs:
        if log["decision"] == "ALLOW":
            allowed += 1
        elif log["decision"] == "BLOCK":
            blocked += 1
    print("\n")
    print("=" * 70)
    print("FIREWALL TRAFFIC STATISTICS")
    print("=" * 70)
    print("Total Traffic :", total)
    print("Allowed :", allowed)
    print("Blocked :", blocked)
    if total > 0:
        allowed_percentage = (allowed / total) * 100
        blocked_percentage = (blocked / total) * 100
        print("Allowed % :", round(allowed_percentage, 2), "%")
        print("Blocked % :", round(blocked_percentage, 2), "%")
    print("=" * 70)
# ============================================================
# 17. MAIN PROGRAM
# ============================================================
print("\n")
print("*" * 90)
print(" FIREWALL CONFIGURATION BASICS")
print(" INCOMING AND OUTGOING TRAFFIC CONTROL")
print("*" * 90)
# ------------------------------------------------------------
# Display status
# ------------------------------------------------------------
display_firewall_status()
# ------------------------------------------------------------
# Display rules
# ------------------------------------------------------------
display_firewall_rules()
# ============================================================
# 18. INBOUND TRAFFIC TESTING
# ============================================================
print("\n")
print("#" * 90)
print(" INBOUND TRAFFIC TESTING")
print("#" * 90)
# Test 1: HTTP
test_traffic(
    direction="INBOUND",
    source="192.168.1.50",
    destination="LOCAL",
    protocol="TCP",
    port=80
)
# Test 2: HTTPS
test_traffic(
    direction="INBOUND",
    source="192.168.1.50",
    destination="LOCAL",
    protocol="TCP",
    port=443
)
# Test 3: SSH from authorized administrator
test_traffic(
    direction="INBOUND",
    source="192.168.1.100",
    destination="LOCAL",
    protocol="TCP",
    port=22
)
# Test 4: SSH from unauthorized system
test_traffic(
    direction="INBOUND",
    source="192.168.1.200",
    destination="LOCAL",
    protocol="TCP",
    port=22
)
# Test 5: Telnet
test_traffic(
    direction="INBOUND",
    source="192.168.1.50",
    destination="LOCAL",
    protocol="TCP",
    port=23
)
# Test 6: FTP
test_traffic(
    direction="INBOUND",
    source="192.168.1.50",
    destination="LOCAL",
    protocol="TCP",
    port=21
)
# Test 7: Application Server
test_traffic(
    direction="INBOUND",
    source="192.168.1.50",
    destination="LOCAL",
    protocol="TCP",
    port=8080
)
# Test 8: Unknown Port
test_traffic(
    direction="INBOUND",
    source="192.168.1.50",
    destination="LOCAL",
    protocol="TCP",
    port=9999
)
# ============================================================
# 19. OUTBOUND TRAFFIC TESTING
# ============================================================
print("\n")
print("#" * 90)
print(" OUTBOUND TRAFFIC TESTING")
print("#" * 90)
# Test 9: DNS UDP
test_traffic(
    direction="OUTBOUND",
    source="LOCAL",
    destination="8.8.8.8",
    protocol="UDP",
    port=53
)
# Test 10: HTTPS
test_traffic(
    direction="OUTBOUND",
    source="LOCAL",
    destination="google.com",
    protocol="TCP",
    port=443
)
# Test 11: Telnet
test_traffic(
    direction="OUTBOUND",
    source="LOCAL",
    destination="192.168.1.50",
    protocol="TCP",
    port=23
)
# Test 12: DNS TCP
test_traffic(
    direction="OUTBOUND",
    source="LOCAL",
    destination="8.8.8.8",
    protocol="TCP",
    port=53
)
# Test 13: Unknown outbound port
test_traffic(
    direction="OUTBOUND",
    source="LOCAL",
    destination="192.168.1.50",
    protocol="TCP",
    port=9999
)
# ============================================================
# 20. ADD NEW FIREWALL RULE
# ============================================================
print("\n")
print("#" * 90)
print(" ADDING NEW FIREWALL RULE")
print("#" * 90)
add_firewall_rule(
    name="Allow Secure Application",
    direction="INBOUND",
    source="192.168.1.150",
    destination="LOCAL",
    protocol="TCP",
    port=8443,
    action="ALLOW"
)
# ============================================================
# 21. TEST NEW RULE
# ============================================================
test_traffic(
    direction="INBOUND",
    source="192.168.1.150",
    destination="LOCAL",
    protocol="TCP",
    port=8443
)
# Unauthorized source
test_traffic(
    direction="INBOUND",
    source="192.168.1.200",
    destination="LOCAL",
    protocol="TCP",
    port=8443
)
# ============================================================
# 22. DISPLAY UPDATED RULES
# ============================================================
display_firewall_rules()
# ============================================================
# 23. DISPLAY LOGS
# ============================================================
display_logs()
# ============================================================
# 24. DISPLAY STATISTICS
# ============================================================
display_statistics()
# ============================================================
# 25. FINAL RESULT
# ============================================================
print("\n")
print("*" * 90)
print(" PRACTICAL COMPLETED")
print("*" * 90)
print("""
Firewall simulation successfully demonstrated:
[1] Firewall status
[2] Firewall rules
[3] Inbound traffic
[4] Outbound traffic
[5] TCP filtering
[6] UDP filtering
[7] Port-based filtering
[8] Source IP filtering
[9] Destination filtering
[10] ALLOW action
[11] BLOCK action
[12] Default-deny policy
[13] Authorized SSH access
[14] Unauthorized SSH blocking
[15] HTTP filtering
[16] HTTPS filtering
[17] DNS filtering
[18] Telnet blocking
[19] FTP blocking
[20] Firewall logging
[21] Firewall statistics
[22] Adding firewall rules
RESULT:
Firewall rules were successfully simulated and network
traffic was classified as ALLOW or BLOCK according to
the configured security policy.
""")
print("*" * 90)



******************************************************************************************
 FIREWALL CONFIGURATION BASICS
 INCOMING AND OUTGOING TRAFFIC CONTROL
******************************************************************************************


FIREWALL STATUS
Firewall Status : ENABLED
Default Policy : BLOCK


CONFIGURED FIREWALL RULES
ID   RULE NAME                  DIRECTION   SOURCE            PROTOCOL  PORT    ACTION    
------------------------------------------------------------------------------------------------------------------------
1    Allow HTTP                 INBOUND     ANY               TCP       80      ALLOW     
2    Allow HTTPS                INBOUND     ANY               TCP       443     ALLOW     
3    Allow SSH from Admin       INBOUND     192.168.1.100     TCP       22      ALLOW     
4    Block Telnet               INBOUND     ANY               TCP       23      BLOCK     
5    Block FTP                  INBOUND     ANY               TCP       21 